In [1]:
%reload_ext dotenv

In [2]:
from pathlib import Path
from dotenv import load_dotenv
env_file = Path("/Users/lvalverdeb/TeamDev/repo-split/boti-data/.env.local")
load_dotenv(dotenv_path=env_file)

True

In [3]:
import os
db_url_async = os.getenv('ASYNC_DB_DSN')
db_url_sync = os.getenv('SYNC_DB_DSN')

In [4]:
from boti_data.helper import DataHelper
config={
    'backend': 'sqlalchemy',
    'connection_url': db_url_async,
    "worker_connection_env_var": "ASYNC_DB_DSN",
    "poolclass": "sqlalchemy.pool.NullPool",
    'query_only': True,
    'table': 'asm_tracking_productos',
    'field_map': {
        'id_track_global': 'global_track_id',
        'id_tipo_producto': 'product_type_id'
    },
    'sticky_filters': {
        'product_type_id': 1,
    },
}

In [5]:
gateway = DataHelper(**config)
columns=['id_producto','cliente_id', 'product_type_id','global_track_id']

In [6]:
result_pandas = await gateway.pandas.aload(global_track_id__in=[1,2,3,4],columns=columns)


In [7]:
result_pandas

,id_producto,cliente_id,product_type_id,global_track_id
0,405720434,139,1,1
1,405728377,139,1,1
2,405787038,139,1,1
3,405787039,139,1,1
4,405787040,139,1,1
...,...,...,...,...
15806,408489624,102,1,1
15807,408489625,102,1,1
15808,408489626,102,1,1
15809,408489627,231,1,1


In [8]:
result_polars = await gateway.polars.aload(global_track_id__in=[1,2,3,4], columns=columns)

In [9]:
result_polars

id_producto,cliente_id,product_type_id,global_track_id
i32,i32,i32,i32
405720434,139,1,1
405728377,139,1,1
405787038,139,1,1
405787039,139,1,1
405787040,139,1,1
…,…,…,…
408489624,102,1,1
408489625,102,1,1
408489626,102,1,1


In [15]:
result_dask = await gateway.dask.aload(global_track_id__in=[1,2,3,4], columns=columns)

In [16]:
result_dask.compute()

,id_producto,cliente_id,product_type_id,global_track_id
0,405720434,139,1,1
1,405728377,139,1,1
2,405787038,139,1,1
3,405787039,139,1,1
4,405787040,139,1,1
...,...,...,...,...
15806,408489624,102,1,1
15807,408489625,102,1,1
15808,408489626,102,1,1
15809,408489627,231,1,1


In [12]:
from boti_data.connection_catalog import S3Catalog
store = S3Catalog("ETL_", env_file=".env")
print(store.storage_path)   # s3://analytics-bucket/raw/events
print(store.ls())

ValidationError: 1 validation error for FilesystemConfig
fs_endpoint
  Value error, fs_endpoint 'http://10.211.55.28:6000' resolves to a private or reserved IP address (10.211.55.28) which is blocked to prevent SSRF attacks. Add the host to boti.core.filesystem.ENDPOINT_ALLOWLIST if this is intentional. [type=value_error, input_value='http://10.211.55.28:6000', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [ ]:
store

In [17]:
from boti_data import ParquetReader

In [18]:
parquet_config = {
    'fs': store.fs(),
    'storage_path': 'dst-etl/bronze/logistics/mobile/gps/',
    'parquet_start_date': '2025-01-01',
    'parquet_end_date': '2026-03-31',
    'partition_on': ['partition_date']
}
parquet_helper = ParquetReader(**parquet_config)

NameError: name 'store' is not defined

In [19]:
result = await parquet_helper.aload(associate_id__in=[27,2285])

NameError: name 'parquet_helper' is not defined

In [20]:
result.compute()

NameError: name 'result' is not defined

In [21]:
parquet_helper.close()

NameError: name 'parquet_helper' is not defined

In [22]:
from boti_dask import (
    UniqueValuesExtractor,
    apply_recommended_dask_config,
    async_safe_compute,
    async_safe_gather,
    async_safe_head,
    async_safe_persist,
    async_safe_wait,
    dask_is_empty,
    dask_is_probably_empty,
    inspect_graph,
    safe_compute,
    safe_gather,
    safe_head,
    safe_persist,
    safe_wait,
)
import dask

graph_metrics = inspect_graph(result_dask)
assert graph_metrics["is_dask"] is True
assert graph_metrics["npartitions"] == result_dask.npartitions

with apply_recommended_dask_config():
    assert dask.config.get("dataframe.shuffle.method") == "tasks"

graph_metrics

{'type': 'DataFrame',
 'is_dask': True,
 'task_count': 5,
 'npartitions': 1,
 'graph_layers': None}

In [23]:
with gateway.session(
    verify_connectivity=True,
    shared=True,
    shared_key="bootstrap-resilience",
    cluster_kwargs={"n_workers": 1, "threads_per_worker": 1, "processes": False, "dashboard_address": ":0"},
):
    dry_run_frame = await gateway.aload(
        global_track_id__in=[1, 2, 3, 4],
        columns=columns,
        return_type="dask",
        persist=True,
        resilient=True,
        diagnostics=True,
        dry_run=True,
    )
    resilient_frame = await gateway.aload(
        global_track_id__in=[1, 2, 3, 4],
        columns=columns,
        return_type="dask",
        persist=True,
        resilient=True,
        diagnostics=True,
    )
    persisted_frame = safe_persist(resilient_frame)
    safe_wait(persisted_frame, timeout=30)
    row_count = safe_compute(resilient_frame["global_track_id"].count())
    gathered_counts = safe_gather([resilient_frame["global_track_id"].count()])
    resilient_preview = safe_head(resilient_frame, n=3)
    async_persisted = await async_safe_persist(resilient_frame)
    await async_safe_wait(async_persisted, timeout=30)
    async_row_count = await async_safe_compute(resilient_frame["global_track_id"].count())
    async_preview = await async_safe_head(resilient_frame, n=2)
    async_gathered = await async_safe_gather([resilient_frame["global_track_id"].count()])
    del async_persisted
    del persisted_frame
    del resilient_frame

assert dry_run_frame.npartitions > 0
assert row_count > 0
assert async_row_count == row_count
assert gathered_counts == [row_count]
assert async_gathered == [row_count]
assert not resilient_preview.empty
assert len(async_preview) == 2

[2026-06-22 08:29:18][INFO][AsyncSqlDatabaseResource] Partitioned SQL plan strategy=offset partitions=1 rows=15811 chunk_size=50000 max_concurrent_fetches=4 use_arrow=True
[2026-06-22 08:29:18][INFO][AsyncSqlDatabaseResource] Partitioned SQL load completed strategy=offset partitions=1 rows=15811 result_partitions=1 elapsed=4.23s
[2026-06-22 08:29:22][INFO][AsyncSqlDatabaseResource] Partitioned SQL plan strategy=offset partitions=1 rows=15811 chunk_size=50000 max_concurrent_fetches=4 use_arrow=True
[2026-06-22 08:29:22][INFO][AsyncSqlDatabaseResource] Partitioned SQL load completed strategy=offset partitions=1 rows=15811 result_partitions=1 elapsed=4.23s


In [24]:
assert dask_is_probably_empty(result_dask) is False
assert dask_is_empty(result_dask) is False

unique_values = await UniqueValuesExtractor().extract_unique_values(
    result_dask,
    "product_type_id",
    "global_track_id",
    limit=10,
)

assert set(unique_values["product_type_id"]) == {1}
assert set(unique_values["global_track_id"]).issubset({1, 2, 3, 4})
unique_values

{'product_type_id': [1], 'global_track_id': [1, 2, 3, 4]}